# SGEMM on a Colab T4
Thin launcher: all logic lives in the repo. Runtime > Change runtime type > **T4 GPU**.

In [ ]:
!nvidia-smi

In [ ]:
# ---- setup: safe to rerun; also pulls your latest push ----
REPO_URL = "https://github.com/ItzSupra13/sgemm.git"   # <- edit me
# Private repo? Store a fine-grained token in Colab Secrets (key icon) named GH_TOKEN, then:
# from google.colab import userdata
# REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/<you>/sgemm.git"

import os
if not os.path.isdir("/content/sgemm"):
    !git clone {REPO_URL} /content/sgemm
%cd /content/sgemm
!git pull --ff-only

# No git yet? Upload sgemm.zip via the Files sidebar instead, then:
# !unzip -qo /content/sgemm.zip -d /content && cd /content/sgemm

In [ ]:
!python run.py gpu info

Compile every `sgemm/gpu/kernels/*.cu` with nvcc (sm_75). Rerun after pulling new kernels: only changed files rebuild.

In [ ]:
!python run.py gpu build

Smoke test first (`--quick`), then the full sweep. Results are resumable: if the session dies, rerun this cell. Add `--kernels cublas_fp32,00_naive` to run just some kernels.

In [ ]:
!python run.py gpu run --quick

In [ ]:
!python run.py gpu run
!python run.py gpu plot

In [ ]:
# ---- show the plots ----
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("sgemm/gpu/plots/*/*.png")):
    print(p)
    display(Image(p))

In [ ]:
# ---- keep the results: Colab disk disappears with the session ----
!zip -qr gpu_out.zip sgemm/gpu/results sgemm/gpu/plots
from google.colab import files
files.download("gpu_out.zip")

# Or copy to Drive instead (commit the CSVs to git from your laptop afterwards):
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r sgemm/gpu/results sgemm/gpu/plots /content/drive/MyDrive/sgemm_gpu/